# Huberman Lab Podcast

### Setup

In [7]:
import os
import pandas as pd
import pickle
from youtube_transcript_api import YouTubeTranscriptApi

In [8]:
ytt_api = YouTubeTranscriptApi()

In [9]:
data = pd.read_csv('data/huberman_videos.csv', encoding='latin1', index_col=False)
data.head()

,id,url,title,video_key
0,0,https://www.youtube.com/watch?v=4b6bwcWK6GE&li...,Welcome to the Huberman Lab Podcast,4b6bwcWK6GE
1,1,https://www.youtube.com/watch?v=H-XfCl-HpRM&li...,How Your Brain Works & Changes,H-XfCl-HpRM
2,2,https://www.youtube.com/watch?v=nm1TxQj9IsQ&li...,Master Your Sleep & Be More Alert When Awake,nm1TxQj9IsQ
3,3,https://www.youtube.com/watch?v=nwSkFq4tyC0&li...,"Using Science to Optimize Sleep, Learning & Me...",nwSkFq4tyC0
4,4,https://www.youtube.com/watch?v=NAATB55oxeQ&li...,"How to Defeat Jet Lag, Shift Work & Sleeplessness",NAATB55oxeQ


### Prepare a utility dict

This will be beneficial when creating the embeddings and allow us to pass the title of each video_id into the embedding module 

In [10]:
title_dict = data.set_index('video_key')['title'].to_dict()
print(f'Video title at index zero:\n{title_dict[data.video_key[:2][0]]}\n')
print(f'Video title at index one:\n{title_dict[data.video_key[:2][1]]}')

Video title at index zero:
Welcome to the Huberman Lab Podcast

Video title at index one:
How Your Brain Works & Changes


In [11]:
with open('data/title_dict.pkl', 'wb') as f:
    pickle.dump(title_dict, f)

### Generate and Store Transcripts

TODO: Update this to utilize `Mongo` opposed to it's current `.txt` storage

In [14]:
import time
import random
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

os.makedirs('data/documents', exist_ok=True)

def get_transcript_with_retry(video_id, max_retries=3, base_delay=2):
    """
    Fetch transcript with exponential backoff retry logic and improved API usage
    """
    for attempt in range(max_retries):
        try:
            # Use list_transcripts to get available transcripts
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            
            # Try to find English transcript (manual first, then auto-generated)
            transcript = None
            
            # First try to get manually created English transcript
            try:
                transcript = transcript_list.find_transcript(['en'])
            except NoTranscriptFound:
                # If no manual English, try auto-generated English
                try:
                    transcript = transcript_list.find_generated_transcript(['en'])
                except NoTranscriptFound:
                    # If no English at all, try any available transcript
                    try:
                        # Get the first available transcript from the list
                        for available_transcript in transcript_list:
                            transcript = available_transcript
                            break
                        
                        if transcript is None:
                            raise NoTranscriptFound(video_id, [], "No transcripts available")
                    except Exception:
                        raise NoTranscriptFound(video_id, [], "No transcripts available")
            
            # Fetch the actual transcript data
            return transcript.fetch()
            
        except (TranscriptsDisabled, VideoUnavailable) as e:
            # These errors won't be resolved by retrying
            raise e
        except Exception as e:
            if attempt == max_retries - 1:
                # Last attempt failed
                raise e
            
            # For rate limiting (429 errors), use longer delays
            if "429" in str(e) or "Too Many Requests" in str(e):
                delay = base_delay * (3 ** attempt) + random.uniform(2, 5)  # Longer delays for rate limits
                print(f"Rate limited on attempt {attempt + 1} for video {video_id}. Retrying in {delay:.2f} seconds...")
            else:
                # For other errors, use shorter delays
                delay = base_delay * (2 ** attempt) + random.uniform(0.5, 1.5)
                print(f"Attempt {attempt + 1} failed for video {video_id}: {str(e)[:100]}... Retrying in {delay:.2f} seconds...")
            
            time.sleep(delay)
    
    return None

# Process videos with improved error handling and rate limiting
successful_downloads = 0
failed_downloads = 0
skipped_existing = 0
rate_limited_count = 0

# Shuffle the video list to distribute load
video_list = list(zip(data['video_key'], data['title']))
random.shuffle(video_list)

print(f"Starting transcript download for {len(video_list)} videos...")
print("=" * 60)

for idx, (video_key, title) in enumerate(video_list):
    filename = f'data/documents/{video_key}.txt'
    
    # Skip if file already exists and has content
    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        print(f"[{idx + 1}/{len(video_list)}] ⏭️  Skipping {video_key} - file already exists")
        skipped_existing += 1
        continue
    
    print(f"[{idx + 1}/{len(video_list)}] 🔄 Processing: {video_key} - {title[:50]}...")
    
    try:
        # Get transcript with retry logic
        fetched_transcript = get_transcript_with_retry(video_key)
        
        if fetched_transcript:
            # Write transcript to file
            with open(filename, 'w', encoding='utf-8') as file:
                for snippet in fetched_transcript:
                    file.write(snippet['text'] + '\n')
            
            successful_downloads += 1
            print(f"[{idx + 1}/{len(video_list)}] ✅ Successfully downloaded transcript for {video_key}")
        else:
            print(f"[{idx + 1}/{len(video_list)}] ❌ No transcript data returned for {video_key}")
            failed_downloads += 1
        
        # Progressive delay - increase delay after every 10 successful downloads to be more respectful
        base_sleep = 1.0 + (successful_downloads // 10) * 0.5
        sleep_time = base_sleep + random.uniform(0.2, 0.8)
        time.sleep(sleep_time)
        
    except TranscriptsDisabled:
        print(f"[{idx + 1}/{len(video_list)}] ❌ Transcripts disabled for video {video_key}")
        failed_downloads += 1
    except VideoUnavailable:
        print(f"[{idx + 1}/{len(video_list)}] ❌ Video {video_key} is unavailable")
        failed_downloads += 1
    except NoTranscriptFound:
        print(f"[{idx + 1}/{len(video_list)}] ❌ No transcript found for video {video_key}")
        failed_downloads += 1
    except Exception as e:
        if "429" in str(e) or "Too Many Requests" in str(e):
            rate_limited_count += 1
            print(f"[{idx + 1}/{len(video_list)}] ⚠️  Rate limited for video {video_key} (#{rate_limited_count})")
            # Extra long pause when rate limited
            time.sleep(10 + random.uniform(5, 15))
        else:
            print(f"[{idx + 1}/{len(video_list)}] ❌ Error processing video {video_key}: {str(e)[:100]}")
        failed_downloads += 1
    
    # Progress update every 25 videos
    if (idx + 1) % 25 == 0:
        current_success_rate = (successful_downloads / max(1, successful_downloads + failed_downloads)) * 100
        print(f"\n📊 Progress Update - Processed {idx + 1}/{len(video_list)} videos")
        print(f"   ✅ Success: {successful_downloads} | ❌ Failed: {failed_downloads} | ⏭️  Skipped: {skipped_existing}")
        print(f"   📈 Current Success Rate: {current_success_rate:.1f}%")
        print("=" * 60)

print(f"\n🎉 === FINAL SUMMARY ===")
print(f"✅ Successful downloads: {successful_downloads}")
print(f"❌ Failed downloads: {failed_downloads}")
print(f"⏭️  Skipped (already exist): {skipped_existing}")
print(f"⚠️  Rate limited attempts: {rate_limited_count}")
print(f"📊 Total processed: {successful_downloads + failed_downloads + skipped_existing}")

if successful_downloads + failed_downloads > 0:
    success_rate = (successful_downloads / (successful_downloads + failed_downloads)) * 100
    print(f"📈 Final Success Rate: {success_rate:.1f}%")

if successful_downloads > 0:
    print(f"🎯 {successful_downloads} transcript files are ready for embedding!")

Starting transcript download for 292 videos...
[1/292] ⏭️  Skipping In9Bq4EJMZw - file already exists
[2/292] ⏭️  Skipping 6P8hrzjnetU - file already exists
[3/292] ⏭️  Skipping kpTJqwIfHcM - file already exists
[4/292] ⏭️  Skipping Mwz8JprPeMc - file already exists
[5/292] ⏭️  Skipping nqNEtdHVUjM - file already exists
[6/292] 🔄 Processing: LTGGyQS1fZE - Science-Based Tools for Increasing Happiness | Hub...
Rate limited on attempt 1 for video LTGGyQS1fZE. Retrying in 4.18 seconds...
Rate limited on attempt 2 for video LTGGyQS1fZE. Retrying in 8.33 seconds...
[6/292] ⚠️  Rate limited for video LTGGyQS1fZE (#1)
[7/292] ⏭️  Skipping 2XGREPnlI8U - file already exists
[8/292] ⏭️  Skipping QYAgf_lfio4 - file already exists
[9/292] 🔄 Processing: SCzecagKBjY - Essentials: Machines, Creativity & Love | Dr. Lex ...
Rate limited on attempt 1 for video SCzecagKBjY. Retrying in 5.44 seconds...
Rate limited on attempt 2 for video SCzecagKBjY. Retrying in 9.10 seconds...
[9/292] ⚠️  Rate limited for